# DopplerGPS: Interactive Acoustic Geolocation Walkthrough
### Developed by Dyne Research ([dyneresearch.com](https://dyneresearch.com))

This interactive notebook demonstrates the complete end-to-end physics and signal processing pipeline of **DopplerGPS**:
1. **Acoustic Doppler Inversion:** Deriving ground observer distance from perceived pitch shifts.
2. **ADS-B Telemetry Processing:** High-throughput spatial querying with DuckDB.
3. **Dual-Event Shared $\Delta t$ Invariant:** Resolving spatial hyperbolas to a single discrete coordinate fix.
4. **Interactive GIS Visualization:** Inspecting converged coordinates on Folium satellite/terrain maps.

> **Responsible Use:** DopplerGPS is published strictly for academic research and defensive OSINT verification. Review `DISCLAIMER.md` before execution.

In [ ]:
# Install dependencies if running in Google Colab
# !pip install git+https://github.com/dyneresearch/DopplerGPS.git

In [ ]:
import os
import tempfile
import pathlib
import numpy as np
import matplotlib.pyplot as plt

from doppler_geo.phase1_ingestion import BoundingBox, ADSBDataLoader, SyntheticDEMHandler
from doppler_geo.phase2_audio import SpectrogramGenerator, inspect_audio
from doppler_geo.phase6_refinement import DopplerGeoPipeline
from doppler_geo.simulation import create_simulation_data
from doppler_geo.coordinates import horizontal_haversine_distance

print("✓ DopplerGPS pipeline modules successfully imported.")

## 1. Synthesize Ground Truth Benchmark Environment
We generate a realistic scenario:
- An observer on elevated terrain ($120\text{m}$ ASL) at coordinates `(40.00000, -75.00000)`.
- Two general aviation propeller flights (Cessna 172 at $130\text{Hz}$ and Piper Cherokee at $110\text{Hz}$) crossing overhead.
- A jet distractor flying at $35,000\text{ft}$ (which the pipeline automatically rejects).

In [ ]:
demo_dir = pathlib.Path("demo_walkthrough_data")
os.makedirs(demo_dir, exist_ok=True)

env = create_simulation_data(demo_dir)
print(f"Ground Truth Observer: Lat {env['true_lat']:.5f}, Lon {env['true_lon']:.5f}")
print(f"ADSB Telemetry Parquet: {env['parquet_path']}")
print(f"Ambient Audio WAV:      {env['audio_path']}")

## 2. Inspect Audio Spectrum & Propeller S-Curves
We use the spectral inspector to visualize the acoustic Doppler sweep.

In [ ]:
diag = inspect_audio(
    audio_path=env['audio_path'],
    output_image_path=str(demo_dir / "audio_inspection.png"),
    min_freq_hz=90.0,
    max_freq_hz=150.0,
)
print(diag.summary())

## 3. Execute End-to-End DopplerGPS Geolocation Pipeline
We run the 6-phase pipeline:
1. Ingestion with DuckDB predicate pushdown.
2. High-resolution STFT generation.
3. Delayed CPA spatial grid filter ($200\text{m}$ coarse).
4. 3D Cubic Spline kinematics and line-of-sight modeling.
5. Locked temporal offset optimization ($\Delta T_{\text{best}}$ invariant).
6. High-resolution micro-grid refinement ($15\text{m}$).

In [ ]:
pipeline = DopplerGeoPipeline(
    dem_handler=env["dem"],
    coarse_resolution_m=100.0,
    micro_resolution_m=15.0,
    micro_radius_m=200.0,
    time_tolerance_s=5.0,
    spec_generator=SpectrogramGenerator(
        n_fft=4096,
        hop_length=256,
        min_freq_hz=90.0,
        max_freq_hz=150.0,
    ),
)

result = pipeline.run(
    adsb_parquet_path=env["parquet_path"],
    measurement_config=env["config"],
    search_bbox=env["bbox"],
    audio_path_or_spec=env["audio_path"],
    output_dir="demo_walkthrough_output",
    top_k=3,
)

## 4. Evaluate Spatial Accuracy & Convergence Metrics

In [ ]:
best_cand = result.top_candidates[0]
dist_error = horizontal_haversine_distance(best_cand.lat, best_cand.lon, env["true_lat"], env["true_lon"])

print("=" * 60)
print("  GEOLOCATION CONVERGENCE SUMMARY")
print("=" * 60)
print(f"True Observer Location:  {env['true_lat']:.5f}, {env['true_lon']:.5f}")
print(f"Estimated Coordinates:   {best_cand.lat:.5f}, {best_cand.lon:.5f}")
print(f"Absolute Spatial Error:  {dist_error:.1f} meters")
print(f"Confidence Assessment:   {result.confidence_status}")
print(f"Locked Clock Offset:     {best_cand.locked_delta_t:+.2f} s")
print(f"Combined Fitness Score:  {best_cand.total_score:.2f}")
print("=" * 60)

## 5. Render Interactive Folium Map
The converged candidate fixes, matched 3D flight trajectories, and regional bounding box are rendered directly below.

In [ ]:
from IPython.display import IFrame
IFrame(result.map_html_path, width="100%", height=550)